# Analyse des Avis et Alertes ANSSI avec Enrichissement des CVE

**TD Final – SUPDEVINCI 2026**

Ce notebook charge les données consolidées (avis/alertes ANSSI + CVE enrichies
via MITRE et EPSS), explore le jeu de données, produit des visualisations, puis
applique et valide un modèle de Machine Learning supervisé et un modèle non
supervisé.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 1. Chargement du fichier CSV produit

In [ ]:
df = pd.read_csv("../data/vulnerabilites_anssi.csv", parse_dates=["date_publication"])
df.shape


In [ ]:
df.head(10)

In [ ]:
df.info()

## 2. Exploration du DataFrame

On regarde la qualité des données (valeurs manquantes, doublons), la
répartition avis/alertes, le nombre de CVE uniques, etc.


In [ ]:
print("Nombre de lignes :", len(df))
print("Nombre de bulletins ANSSI uniques :", df["id_anssi"].nunique())
print("Nombre de CVE uniques :", df["cve"].nunique())
print("Nombre d'éditeurs uniques :", df["editeur"].nunique())
print("Nombre de produits uniques :", df["produit"].nunique())


In [ ]:
df.isna().mean().sort_values(ascending=False).to_frame("taux_valeurs_manquantes")


In [ ]:
df["type_bulletin"].value_counts()


In [ ]:
df["base_severity"].value_counts(dropna=False)


In [ ]:
df[["cvss_score", "epss_score"]].describe()


## 3. Visualisations

### 3.1 Histogramme des scores CVSS

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["cvss_score"].dropna(), bins=20, kde=True, color="steelblue")
plt.title("Distribution des scores CVSS")
plt.xlabel("Score CVSS")
plt.ylabel("Nombre de CVE")
plt.show()


### 3.2 Répartition par niveau de sévérité (Base Severity)

In [ ]:
plt.figure(figsize=(6, 6))
df["base_severity"].value_counts().plot.pie(autopct="%1.1f%%", startangle=90)
plt.ylabel("")
plt.title("Répartition des vulnérabilités par sévérité CVSS")
plt.show()


### 3.3 Diagramme circulaire des types de vulnérabilités (CWE)

In [ ]:
top_cwe = df["cwe"].value_counts().head(10)
plt.figure(figsize=(7, 7))
top_cwe.plot.pie(autopct="%1.1f%%", startangle=90)
plt.ylabel("")
plt.title("Top 10 des types de vulnérabilités (CWE)")
plt.show()


### 3.4 Distribution des scores EPSS (probabilité d'exploitation)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["epss_score"].dropna(), bins=30, color="indianred")
plt.title("Distribution des scores EPSS")
plt.xlabel("Score EPSS (probabilité d'exploitation)")
plt.ylabel("Nombre de CVE")
plt.show()


### 3.5 Classement des éditeurs les plus affectés

In [ ]:
top_vendors = df["editeur"].value_counts().head(15)
plt.figure(figsize=(9, 6))
sns.barplot(x=top_vendors.values, y=top_vendors.index, palette="viridis")
plt.title("Top 15 des éditeurs les plus affectés (nombre de CVE)")
plt.xlabel("Nombre de CVE")
plt.ylabel("Éditeur")
plt.show()


### 3.6 Classement des produits les plus affectés

In [ ]:
top_products = df["produit"].value_counts().head(15)
plt.figure(figsize=(9, 6))
sns.barplot(x=top_products.values, y=top_products.index, palette="mako")
plt.title("Top 15 des produits les plus affectés (nombre de CVE)")
plt.xlabel("Nombre de CVE")
plt.ylabel("Produit")
plt.show()


### 3.7 Heatmap de corrélation entre CVSS et EPSS

In [ ]:
corr = df[["cvss_score", "epss_score"]].corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Corrélation entre score CVSS et score EPSS")
plt.show()


### 3.8 Nuage de points Score CVSS vs Score EPSS

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="cvss_score", y="epss_score", hue="base_severity", alpha=0.7)
plt.title("Score CVSS vs Score EPSS")
plt.xlabel("Score CVSS")
plt.ylabel("Score EPSS")
plt.show()


### 3.9 Courbe cumulative des vulnérabilités détectées dans le temps

In [ ]:
ts = (
    df.dropna(subset=["date_publication"])
    .drop_duplicates("id_anssi")
    .sort_values("date_publication")
)
ts["cumul"] = range(1, len(ts) + 1)

plt.figure(figsize=(10, 5))
plt.plot(ts["date_publication"], ts["cumul"])
plt.title("Évolution cumulative du nombre de bulletins ANSSI publiés")
plt.xlabel("Date de publication")
plt.ylabel("Nombre cumulé de bulletins")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 3.10 Boxplot des scores CVSS par éditeur (top 10)

In [ ]:
top10_vendors = df["editeur"].value_counts().head(10).index
subset = df[df["editeur"].isin(top10_vendors)]

plt.figure(figsize=(10, 6))
sns.boxplot(data=subset, x="editeur", y="cvss_score")
plt.title("Dispersion des scores CVSS pour les 10 éditeurs les plus affectés")
plt.xticks(rotation=45, ha="right")
plt.xlabel("Éditeur")
plt.ylabel("Score CVSS")
plt.tight_layout()
plt.show()


### 3.11 Nombre de vulnérabilités par éditeur, par type de bulletin

In [ ]:
cross = df[df["editeur"].isin(top10_vendors)].groupby(["editeur", "type_bulletin"]).size().unstack(fill_value=0)
cross.plot(kind="bar", stacked=True, figsize=(10, 6), colormap="Set2")
plt.title("Nombre de vulnérabilités par éditeur, par type de bulletin (avis/alerte)")
plt.xlabel("Éditeur")
plt.ylabel("Nombre de CVE")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 4. Préparation des données pour le Machine Learning

On construit un jeu de features numériques/catégorielles à partir des
colonnes enrichies (CVSS, CWE, EPSS, éditeur...). On élimine les lignes sans
score CVSS ou EPSS (CVE non trouvées dans MITRE/FIRST), nécessaires pour
les deux approches ML.


In [ ]:
ml_df = df.dropna(subset=["cvss_score", "epss_score", "cwe"]).copy()
ml_df = ml_df.drop_duplicates(subset=["cve"])
print("Nombre de CVE exploitables pour le ML :", len(ml_df))
ml_df[["cve", "cvss_score", "base_severity", "cwe", "epss_score"]].head()


## 5. Modèle non supervisé : Clustering des vulnérabilités

Objectif : regrouper les CVE en clusters homogènes à partir de leur score
CVSS et de leur score EPSS, afin d'identifier des profils de risque
(ex : faible gravité/faible exploitation, forte gravité/forte exploitation...).

On utilise **KMeans**, avec une normalisation préalable des features et une
recherche du nombre de clusters via le score de silhouette.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_cluster = ml_df[["cvss_score", "epss_score"]].values
X_scaled = StandardScaler().fit_transform(X_cluster)

scores = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

plt.figure(figsize=(7, 4))
plt.plot(list(scores.keys()), list(scores.values()), marker="o")
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Score de silhouette")
plt.title("Choix du nombre de clusters (KMeans)")
plt.show()

best_k = max(scores, key=scores.get)
print("Meilleur k :", best_k, "- silhouette :", scores[best_k])


In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
ml_df["cluster"] = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=ml_df, x="cvss_score", y="epss_score", hue="cluster", palette="tab10")
plt.title(f"Clustering KMeans des vulnérabilités (k={best_k})")
plt.xlabel("Score CVSS")
plt.ylabel("Score EPSS")
plt.show()

ml_df.groupby("cluster")[["cvss_score", "epss_score"]].mean()


**Validation du modèle non supervisé** : le score de silhouette (calculé
ci-dessus pour plusieurs valeurs de k) mesure la cohérence interne des
clusters (entre -1 et 1, plus il est proche de 1 meilleure est la séparation).
Le k retenu est celui qui maximise ce score.


## 6. Modèle supervisé : Prédiction de la sévérité (Base Severity)

Objectif : prédire la catégorie de sévérité CVSS (`LOW`, `MEDIUM`, `HIGH`,
`CRITICAL`) d'une vulnérabilité à partir de l'EPSS et du type de faiblesse
(CWE), c'est-à-dire sans connaître directement le score CVSS — utile pour
estimer rapidement la criticité d'une faille nouvellement publiée avant
l'attribution officielle d'un score CVSS détaillé.


In [ ]:
ml_sup = ml_df.dropna(subset=["base_severity"]).copy()
ml_sup["base_severity"].value_counts()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Encodage du CWE (catégoriel) et de la cible
cwe_encoder = LabelEncoder()
ml_sup["cwe_encoded"] = cwe_encoder.fit_transform(ml_sup["cwe"].astype(str))

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(ml_sup["base_severity"])
X = ml_sup[["epss_score", "cwe_encoded"]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_encoder.classes_, yticklabels=target_encoder.classes_)
plt.xlabel("Prédiction")
plt.ylabel("Réalité")
plt.title("Matrice de confusion - Prédiction de la sévérité CVSS")
plt.show()


In [ ]:
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.plot.bar(figsize=(6, 4), title="Importance des features (RandomForest)")
plt.ylabel("Importance")
plt.show()


**Validation du modèle supervisé** : on utilise un split train/test
stratifié (75/25) pour préserver la distribution des classes, puis on évalue
l'accuracy, la précision/rappel/F1 par classe (classification_report) et la
matrice de confusion. L'importance des features confirme la contribution du
score EPSS et du type de CWE à la prédiction.


## 7. Conclusion

- Les visualisations mettent en évidence les éditeurs et produits les plus
  exposés, ainsi que les types de vulnérabilités (CWE) les plus fréquents.
- Le clustering non supervisé permet de distinguer des profils de risque
  (gravité vs probabilité d'exploitation), utiles pour prioriser les actions
  de remédiation.
- Le modèle supervisé montre qu'il est possible d'estimer la sévérité d'une
  vulnérabilité à partir de l'EPSS et du CWE, avec une performance qui peut
  servir de base à un système d'alerte précoce avant l'attribution complète
  d'un score CVSS.
- Ces résultats alimentent directement la logique de génération d'alertes
  personnalisées du module `src/alerting.py` (cf. Étape 7 du sujet).
